In [0]:
dbutils.widgets.removeAll()

In [0]:
# DBTITLE 1, Define Global Pipeline Parameters
dbutils.widgets.text(
    "CatalogPath", 
    "claimsprocessing.bronze.quality_measures", 
    "Target Bronze Catalog Table Path"
)
catalog_path = dbutils.widgets.get("CatalogPath")

In [0]:
# DBTITLE 2, Environment and Dependency Setup
import os
import sys
import json
import shutil
from pathlib import Path

os.sync()

ROOT_DIR = Path("/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing")
sys.path.append(str(ROOT_DIR))

In [0]:
# DBTITLE 3, File Lifecycle Management Utilities
def move_file(src_path: Path, target_dir: Path) -> Path:
    """Moves a file to a target directory cleanly, ensuring the directory exists."""
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / src_path.name
    if target_path.exists():
        target_path.unlink() 
    shutil.move(str(src_path), str(target_path))
    return target_path

In [0]:
def finalize_file_tracking(file_path: Path, base_source_dir: Path, success: bool):
    """Moves the file out of 'inprogress' ONLY after all downstream stages complete."""
    try:
        if success:
            print(f"--> [Success] Moving file from inprogress to processed: {file_path.name}")
            move_file(file_path, base_source_dir / "processed")
        else:
            print(f"--> [Failure] Moving file from inprogress to failed triage: {file_path.name}")
            move_file(file_path, base_source_dir / "failed")
    except Exception as e:
        print(f"Failed to update final lifecycle directory state: {e}")

In [0]:
def stage_to_inprogress(incoming_file_path: Path, base_source_dir: Path) -> Path:
    """Instantly isolates the file from pending into inprogress at pipeline start."""
    try:
        if not incoming_file_path.exists():
            raise FileNotFoundError(f"Input file missing: {incoming_file_path}")
        
        print(f"--> [Staging] Isolating file from pending to inprogress: {incoming_file_path.name}")
        active_file_path = move_file(incoming_file_path, base_source_dir / "inprogress")
        return active_file_path
    except Exception as e:
        print(f"Critical error during initial staging for {incoming_file_path.name}: {e}")
        raise

In [0]:
# DBTITLE 4, Notebook Execution Functions
def trigger_bronze_ingestion(csv_file_path: Path, target_catalog: str):
    """Triggers Bronze transformation processing against the file inside inprogress."""
    bronze_notebook_path = f"{ROOT_DIR}/DimGapsInCare/Bronze/csv2parquet"
    print(f"--> Running Bronze Ingestion for: inprogress/{csv_file_path.name}")
    
    bronze_arguments = {
        "csv_path": str(csv_file_path),
        "catalog_path": target_catalog
    }
    dbutils.notebook.run(bronze_notebook_path, 600, arguments=bronze_arguments)

In [0]:
def trigger_silver_processing(config_path: str):
    """Executes the generalized Silver layer metadata calculation loop."""
    silver_notebook_path = f"{ROOT_DIR}/DimGapsInCare/Silver/Notebooks/GenericSubGroupProcessing"
    print(f"--> Running Silver Layer Transformations...")
    
    # Removed ClientContainer argument since it's hardcoded inside JSON
    payload_arguments = {
        "SubGroupConfigPath": config_path
    }
    result = dbutils.notebook.run(silver_notebook_path, 1200, arguments=payload_arguments)
    print(f"    Silver Process Response: {result}")

In [0]:
# DBTITLE 5, Main Sequenced Execution Pipeline Loop
def main():
    base_source_dir = ROOT_DIR / "source/DimQualityEvent"
    pending_dir = base_source_dir / "pending"
    gaps_in_care_config = f"{ROOT_DIR}/DimGapsInCare/Silver/Config/GapsInCare.json"
    
    if not pending_dir.exists():
        print(f"Target entrypoint directory {pending_dir} does not exist.")
        return
        
    incoming_files = [f for f in pending_dir.iterdir() if f.is_file() and not f.name.startswith('.')]
    if not incoming_files:
        print("No new files found in pending.")
        return

    print(f"Discovered {len(incoming_files)} pending files to process sequentially...")

    for file_path in incoming_files:
        active_tracked_file = None
        try:
            print(f"\n=======================================================")
            print(f"STARTING LIFECYCLE FOR TARGET FILE: {file_path.name}")
            print(f"=======================================================")

            # Step 1: Instantly move file from pending to inprogress
            active_tracked_file = stage_to_inprogress(file_path, base_source_dir)
            
            # Step 2: Run Bronze Layer using the inprogress file path
            trigger_bronze_ingestion(active_tracked_file, catalog_path)
            
            # Step 3: Run Silver Layer metadata execution (without ClientContainer placeholder)
            trigger_silver_processing(gaps_in_care_config)
            
            # Step 4: After ALL processing succeeds, move from inprogress to processed
            finalize_file_tracking(active_tracked_file, base_source_dir, success=True)
            print(f"Pipeline completely finished processing: {file_path.name}\n")
            
        except Exception as e:
            print(f"CRITICAL: Execution faulted during the lifecycle process. Error: {e}")
            target_to_fail = active_tracked_file if active_tracked_file else file_path
            finalize_file_tracking(target_to_fail, base_source_dir, success=False)
            continue

In [0]:
if __name__ == "__main__":
    main()